# Урок 4. CSV, Excel, JSON и первый проект

**Курс:** pandas от нуля до профессионала · **Модуль 01** · Урок 4 из 4

**Как работать:** внимательно читайте теорию, запускайте демонстрационные ячейки, затем выполняйте задания без копирования примеров. У каждого задания есть отдельная ячейка проверки на показанных исходных данных.

> Для заданий вам понадобятся только знания из этого и предшествующих уроков. Интернет не требуется.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
from tests.notebook_checks import expected_for, check_result, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Что такое табличный файл

**CSV** — текстовые строки с разделителями, обычно запятыми или точками с запятой. CSV не хранит форматирование, несколько листов или типы данных так, как это делает Excel. **XLSX** — формат электронной таблицы с листами. **JSON** — текстовый формат, часто используемый API; его данные могут быть вложенными.

В папке `data/` уже есть учебные файлы: `store_orders.csv`, `store_orders_semicolon.csv`, `store_products.xlsx`, `store_clients.json`. Они намеренно содержат разные варианты форматов и несколько пропущенных значений.

## 2. Чтение CSV

```python
import pandas as pd
from pathlib import Path
path = Path("data") / "store_orders.csv"
df = pd.read_csv(path)
```

Здесь используется путь относительно **корня проекта**, если VS Code запущен из него. Если файл не найден, проверьте текущую рабочую директорию: `Path.cwd()`. Абсолютные пути с конкретного компьютера хуже переносятся на другую машину.

Основные параметры: `sep=";"` для разделителя, `encoding="utf-8"` для кодировки, `usecols=[...]` для нужных столбцов, `dtype={...}` для типов, `parse_dates=[...]` для дат.

Столбец, похожий на число, например почтовый индекс `"00123"`, может потерять ведущие нули при автоматическом определении типа. Для идентификаторов указывайте `dtype={"postal_code": "string"}`.

## 3. Чтение и сохранение Excel

```python
products = pd.read_excel("data/store_products.xlsx", sheet_name="Products")
products.to_excel("output/products_copy.xlsx", index=False)
```

Для чтения `.xlsx` обычно нужен пакет `openpyxl`. `sheet_name` выбирает лист. `index=False` отключает запись индекса как отдельного столбца — при обычном экспорте аналитической таблицы это часто желательно. При работе с Excel помните, что отображаемый в Excel формат и фактическое значение ячейки — разные вещи.

Не записывайте в исходный учебный файл. Используйте папку `output/`. Она создаётся автоматически в первой кодовой ячейке урока.

## 4. Чтение и сохранение JSON

`pd.read_json(path)` подходит для некоторых табличных JSON-структур. Для вложенных словарей и списков чаще удобнее сначала прочитать файл стандартным модулем `json`, затем создать DataFrame.

```python
import json
from pathlib import Path
rows = json.loads(Path("data/example.json").read_text(encoding="utf-8"))
table = pd.DataFrame(rows)
```

Универсального преобразования произвольного JSON в таблицу не существует: нужно решить, что считать строкой и колонкой. В следующих модулях изучим `json_normalize()` для вложенных объектов.

Экспорт CSV: `df.to_csv("output/result.csv", index=False)`; импорт обратно можно проверить `pd.read_csv(...)`. Не ожидайте абсолютного сохранения всех типов при круговом чтении CSV: текстовый формат не хранит их явно.

## 5. Мини-проект: данные магазина

Вы получили заказы и каталог товаров. На данном этапе **не нужно** делать JOIN — это будет отдельная тема. Ваша задача: загрузить заказы, рассмотреть структуру, выявить возможные проблемы, вычислить стоимость каждой строки заказа как `quantity * unit_price` и сохранить только нужные колонки в новый CSV.

**Производственный принцип:** разделяйте чтение и преобразование. Функция обработки должна принимать DataFrame и возвращать DataFrame; чтение файлов и сохранение результатов лучше размещать отдельно. Так проще тестировать преобразование на различных входных данных.

Перед выполнением убедитесь, что понимаете разницу между **данными**, **путём к файлу** и **функцией преобразования**.

## Дополнительный разбор: надёжный ввод-вывод
CSV — это текст с правилами: разделители, кавычки, кодировка и заголовки. Идентификаторы с нулями (`00123`) следует читать строками, иначе нули могут исчезнуть. `read_csv` использует запятую по умолчанию, но российские выгрузки часто применяют точку с запятой (`sep=';'`). У Excel могут быть несколько листов; используйте `sheet_name` и проверяйте имена колонок.

JSON — не обязательно таблица: список словарей удобно конвертировать в DataFrame, вложенные объекты разберём позже через `json_normalize`. При экспорте для делового обмена часто нужно `index=False`: иначе индекс становится отдельным столбцом. Всегда проверяйте сохранённый файл повторным чтением, особенно если дальнейшая обработка автоматизирована.

Если функция работает с файлом, передавайте путь аргументом. Не вшивайте путь к собственной папке пользователя: программа должна выполняться в VS Code и на другом компьютере.

---
## Практические задания

**Правила:** выполните ячейку исходных данных, затем пишите обычный код в ячейке решения. Сохраните полученный результат в переменной `result`. Сразу под ней находится **ожидаемый вывод**. Сравните его со своим результатом, затем запустите автопроверку. Функции объявлять не нужно.

**Почему задания отличаются от теории:** в теории показаны простые иллюстрации методов, а здесь другая предметная область и дополнительные условия. Автопроверка сравнивает ваше решение с результатом на приведённых данных.

### Задание 1 · m1_16

Функция принимает путь `path` к CSV с разделителем `;`. Получите DataFrame; колонку `order_id` загрузите как строковый тип, чтобы сохранить ведущие нули.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `path` — путь к файлу (`pathlib.Path`), создаётся отдельной ячейкой ниже.


**Выполнение:** используйте готовые переменные `path`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_16: используются переменные path

path = ROOT / "output" / "examples" / "m1_16_example.csv"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text("order_id;quantity\n00102;4\n00999;2\n", encoding="utf-8")


# Снимок правильного результата для автопроверки.
_expected_m1_16 = expected_for("m1_16", path=path)


In [ ]:
# m1_16: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: path
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 2 столбца; колонки: `order_id`, `quantity`.

```text
  order_id  quantity
0    00102         4
1    00999         2
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_16", result, _expected_m1_16)

### Задание 2 · m1_17

Прочитайте Excel по пути `path`, лист с именем `sheet`. Получите DataFrame.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `path` — путь к файлу (`pathlib.Path`), создаётся отдельной ячейкой ниже.
- `sheet` — входной параметр (значение видно в примере); в этом примере: `'Products'`.


**Выполнение:** используйте готовые переменные `path`, `sheet`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_17: используются переменные path, sheet

path = ROOT / "output" / "examples" / "m1_17_example.xlsx"
path.parent.mkdir(parents=True, exist_ok=True)
path_table = pd.DataFrame({
    'code': [3, 8],
    'price': [15.0, 65.0],
})
with pd.ExcelWriter(path) as writer:
    path_table.to_excel(writer, sheet_name="Products", index=False)

sheet = 'Products'


# Снимок правильного результата для автопроверки.
_expected_m1_17 = expected_for("m1_17", path=path, sheet=sheet)


In [ ]:
# m1_17: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: path, sheet
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 2 столбца; колонки: `code`, `price`.

```text
   code  price
0     3     15
1     8     65
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_17", result, _expected_m1_17)

### Задание 3 · m1_18

Прочитайте JSON-файл по пути `path`, который содержит **список объектов**. Получите DataFrame, сохраняя порядок записей.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `path` — путь к файлу (`pathlib.Path`), создаётся отдельной ячейкой ниже.


**Выполнение:** используйте готовые переменные `path`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_18: используются переменные path

path = ROOT / "output" / "examples" / "m1_18_example.json"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('[{"id": 1, "city": "Москва"}, {"id": 2, "city": "Казань"}]', encoding="utf-8")


# Снимок правильного результата для автопроверки.
_expected_m1_18 = expected_for("m1_18", path=path)


In [ ]:
# m1_18: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: path
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 2 столбца; колонки: `id`, `city`.

```text
   id    city
0   1  Москва
1   2  Казань
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_18", result, _expected_m1_18)

### Задание 4 · m1_19

Получите таблицу с колонками `order_id`, `product_id`, `line_total`. Последняя — произведение `quantity` и `unit_price`. Исходную таблицу не меняйте.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `product_id`, `quantity`, `unit_price`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_19: используются переменные df

df = pd.DataFrame({
    'order_id': ['a', 'b'],
    'product_id': [104, 101],
    'quantity': [2, 5],
    'unit_price': [7.0, 3.0],
})

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m1_19 = expected_for("m1_19", df=df)


In [ ]:
# m1_19: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 3 столбца; колонки: `order_id`, `product_id`, `line_total`.

```text
  order_id  product_id  line_total
0        a         104        14.0
1        b         101        15.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_19", result, _expected_m1_19)

### Задание 5 · m1_20

Функция принимает DataFrame и путь `path`. Сохраните CSV без индекса в UTF-8 и получите `path`. Столбцы должны остаться в исходном порядке.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `name`, `qty`.
- `path` — путь к файлу (`pathlib.Path`), создаётся отдельной ячейкой ниже.


**Выполнение:** используйте готовые переменные `df`, `path`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_20: используются переменные df, path

df = pd.DataFrame({
    'name': ['Ручка', 'Линейка'],
    'qty': [2, 4],
})

path = ROOT / "output" / "examples" / "m1_20_example.csv"
path.parent.mkdir(parents=True, exist_ok=True)
# ВАЖНО: этот путь указывает на БУДУЩИЙ результат. Сохранить файл должна ваша функция.

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m1_20 = expected_for("m1_20", df=df, path=path)


In [ ]:
# m1_20: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, path
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Верните тот же объект пути `path`, который был передан в функцию; файл должен быть сохранён на диск.

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_20", result, _expected_m1_20)

---
## Рефлексия

1. Какие методы и свойства вы сегодня применяли?
2. Какие ошибки возникали и почему?
3. Что изменится, если входная таблица содержит больше строк или иной порядок значений?

Переходите к следующему уроку, когда все задания текущего прошли проверку.